In [0]:
import boto3
import json
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

# read secrets
access_key  = dbutils.secrets.get(scope="itsm-pipeline", key="AWS_ACCESS_KEY_ID")
secret_key  = dbutils.secrets.get(scope="itsm-pipeline", key="AWS_SECRET_ACCESS_KEY")
bucket_name = dbutils.secrets.get(scope="itsm-pipeline", key="AWS_BUCKET_NAME")

# download the file from S3 using boto3
s3 = boto3.client(
    "s3",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="us-east-1"
)

obj = s3.get_object(
    Bucket=bucket_name,
    Key=f"raw/incidents/{today}/incidents_{today}.json"
)
incidents = json.loads(obj["Body"].read().decode("utf-8"))
print(f"Downloaded {len(incidents)} incidents from S3")



In [0]:
# list all files in your S3 bucket under raw/incidents/
response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix="raw/incidents/"
)

for obj in response.get("Contents", []):
    print(obj["Key"])



In [0]:
print(incidents)

In [0]:
# Cell 2: clean the raw data — flatten nested fields AND fix bad values
cleaned = []

for row in incidents:
    # clean resolved_at — replace "-" with None before Spark sees it
    resolved = row.get("resolved_at")
    if not resolved or resolved == "-":
        resolved = None

    cleaned.append({
        "number":            row.get("number"),
        "short_description": row.get("short_description"),
        "priority":          row.get("priority"),
        "state":             row.get("state"),
        "category":          row.get("category"),
        "opened_at":         row.get("opened_at") or None,
        "resolved_at":       resolved,
        "closed_at":         row.get("closed_at") or None,
        "sys_updated_on":    row.get("sys_updated_on") or None,
        "assignment_group":  row["assignment_group"].get("display_value")
                             if isinstance(row.get("assignment_group"), dict)
                             else None,
        "assigned_to":       row["assigned_to"].get("display_value")
                             if isinstance(row.get("assigned_to"), dict)
                             else None,
    })

df = spark.createDataFrame(cleaned)
print(f"Row count: {df.count()}")
display(df)

In [0]:
# add two calculated columns
# MTTR = how many hours between opened_at and resolved_at
# SLA breach risk = a score based on priority (3=critical, 0=low)

from pyspark.sql.functions import (
    unix_timestamp, round, when, col
)

# cast timestamps so Spark can do date math on them
df = df.withColumn("opened_at",   col("opened_at").cast("timestamp"))
df = df.withColumn("resolved_at", col("resolved_at").cast("timestamp"))

# calculate resolution time in hours
df = df.withColumn(
    "resolution_hours",
    round(
        (unix_timestamp("resolved_at") - unix_timestamp("opened_at")) / 3600,
        1
    )
)

# SLA breach risk score based on priority
df = df.withColumn(
    "sla_breach_risk",
    when(col("priority") == "1 - Critical", 3)
    .when(col("priority") == "2 - High",    2)
    .when(col("priority") == "3 - Moderate", 1)
    .otherwise(0)
)

display(df)

In [0]:
# save the transformed data as a managed Delta table
# dbt will read from this table in the next phase

spark.sql("CREATE DATABASE IF NOT EXISTS itsm_pipeline")

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("itsm_pipeline.stg_incidents")

print("Saved to itsm_pipeline.stg_incidents")

# verify it worked
spark.sql("SELECT COUNT(*) as total_rows FROM itsm_pipeline.stg_incidents").show()
spark.sql("""
    SELECT number, priority, state, resolution_hours, sla_breach_risk
    FROM itsm_pipeline.stg_incidents
    LIMIT 5
""").show()

In [0]:
import boto3
import io
import pandas as pd
from datetime import datetime

access_key  = dbutils.secrets.get(scope="itsm-pipeline", key="AWS_ACCESS_KEY_ID")
secret_key  = dbutils.secrets.get(scope="itsm-pipeline", key="AWS_SECRET_ACCESS_KEY")
bucket_name = dbutils.secrets.get(scope="itsm-pipeline", key="AWS_BUCKET_NAME")

# read from managed table
df_export = spark.sql("""
    SELECT
        number,
        short_description,
        priority,
        state,
        category,
        assignment_group,
        assigned_to,
        CAST(opened_at AS STRING)      as opened_at,
        CAST(resolved_at AS STRING)    as resolved_at,
        CAST(closed_at AS STRING)      as closed_at,
        CAST(sys_updated_on AS STRING) as sys_updated_on,
        CAST(resolution_hours AS DOUBLE) as resolution_hours,
        CAST(sla_breach_risk AS INT)   as sla_breach_risk
    FROM itsm_pipeline.stg_incidents
""")

# convert to pandas
pandas_df = df_export.toPandas()

# strip all metadata that Spark attaches — this is what causes PlanMetrics error
# we do this by recreating the DataFrame from scratch with no hidden metadata
clean_df = pd.DataFrame(pandas_df.to_dict(orient="records"))

print(f"Rows: {len(clean_df)}")

# write to parquet buffer
buffer = io.BytesIO()
clean_df.to_parquet(buffer, index=False, engine="pyarrow")
buffer.seek(0)

# upload to S3
s3 = boto3.client(
    "s3",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="us-east-1"
)

today = datetime.now().strftime("%Y-%m-%d")
s3_key = f"staging/incidents/{today}/stg_incidents.parquet"

s3.put_object(
    Bucket=bucket_name,
    Key=s3_key,
    Body=buffer.getvalue()
)

print(f"Exported to s3://{bucket_name}/{s3_key}")